# Proyecto 2 — Análisis Exploratorio de Datos

## Reto #11: Predicción de compradores recurrentes: cuestionar la línea base

**CC3084 – Data Science | Universidad del Valle de Guatemala | Semestre II – 2026**

**David Dominguez - 23712**
**Gabriel Bran - 23590**
**Luis Padilla - 23663**


---

### Alcance

Este notebook contiene un **análisis exploratorio de datos (EDA)** completo sobre el conjunto de datos de la competencia *Repeat Buyers Prediction* de la plataforma Tianchi (Alibaba). **No se entrenan modelos predictivos** en este proyecto.

### Pregunta de análisis

> ¿Qué patrones de comportamiento de compra y características demográficas se asocian con que un usuario de la plataforma Tmall vuelva a comprar a un mismo vendedor?

### Enlace al reto original

[Tianchi — Repeat Buyers Prediction](https://tianchi.aliyun.com/competition/entrance/231576/information)

## 2. Configuración

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
import gc

# Suprimir advertencias innecesarias
warnings.filterwarnings('ignore')

# Semilla para reproducibilidad
np.random.seed(42)

# --- Rutas del proyecto, resueltas desde la raíz real del repositorio ---
def _find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'data' / 'data_format1').exists():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / 'data' / 'data_format1'
TRAIN_PATH = DATA_DIR / 'train_format1.csv'
TEST_PATH = DATA_DIR / 'test_format1.csv'
USER_INFO_PATH = DATA_DIR / 'user_info_format1.csv'
USER_LOG_PATH = DATA_DIR / 'user_log_format1.csv'
SAMPLE_SUB_PATH = PROJECT_ROOT / 'data' / 'sample_submission.csv'

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TAB_DIR = PROJECT_ROOT / 'outputs' / 'tables'

# --- Crear directorios de salida ---
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# --- Configuración visual ---
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style('whitegrid')
PALETTE = sns.color_palette('Set2')
COLOR_0 = PALETTE[1]  # No recurrente
COLOR_1 = PALETTE[0]  # Recurrente
labels_map = {0: 'No recurrente', 1: 'Recurrente'}


def normalize_age_range(series: pd.Series) -> pd.Series:
    """Conserva 0 como desconocido y colapsa 7/8 en 50+."""
    return series.fillna(0).replace({8: 7}).astype(int)


age_labels = {
    0: 'Desconocido',
    1: '<18',
    2: '18-24',
    3: '25-29',
    4: '30-34',
    5: '35-39',
    6: '40-49',
    7: '50+',
}
gender_labels = {0: 'Femenino', 1: 'Masculino', 2: 'Desconocido'}

print("Configuración completada.")

Configuración completada.


## 3. Verificación de archivos

In [2]:
archivos = {
    'Entrenamiento': TRAIN_PATH,
    'Prueba': TEST_PATH,
    'Información de usuarios': USER_INFO_PATH,
    'Registro de actividad': USER_LOG_PATH,
    'Envío de muestra': SAMPLE_SUB_PATH,
}

print(f"{'Archivo':<30} {'Existe':<8} {'Tamaño (MB)':>12}")
print("-" * 55)
for nombre, ruta in archivos.items():
    existe = os.path.isfile(ruta)
    if existe:
        tam = os.path.getsize(ruta) / (1024 ** 2)
        print(f"{nombre:<30} {'Sí':<8} {tam:>10.1f} MB")
    else:
        print(f"{nombre:<30} {'NO':<8} {'---':>12}")

print()
print(" NOTA: El archivo de registro de actividad (user_log_format1.csv)")
print("   ocupa aproximadamente 1.9 GB y contiene ~55 millones de filas.")
print("   Se procesará mediante lectura por fragmentos (chunks).")

Archivo                        Existe    Tamaño (MB)
-------------------------------------------------------
Entrenamiento                  Sí              3.4 MB
Prueba                         Sí              3.1 MB
Información de usuarios        Sí              4.3 MB
Registro de actividad          Sí           1821.7 MB
Envío de muestra               Sí              3.9 MB

 NOTA: El archivo de registro de actividad (user_log_format1.csv)
   ocupa aproximadamente 1.9 GB y contiene ~55 millones de filas.
   Se procesará mediante lectura por fragmentos (chunks).
